# 01 – Data audit 

Goal: find every data problem in the four raw files, count how many rows it affects,
and decide whether to **fix, flag or drop** it, with a reason.
Findings are collected in an issue log and summarized in `findings.md`.

Raw files are never edited; every change is made in code so it can be reproduced
and later applied inside the cleaning transform.

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
FILES = {
    "applications": "applications.csv",
    "bureau": "credit_bureau.csv",
    "servicing": "loan_servicing.csv",
    "holdout": "holdout_applications.csv",
}

def load_raw(raw_dir=RAW):
    """Read each raw file; force id to text so ids like '0123456789' keep their leading zero."""
    return {name: pd.read_csv(raw_dir / fname, dtype={"id": str}, low_memory=False)
            for name, fname in FILES.items()}

data = load_raw()
apps, bureau, serv, hold = (data[k] for k in ["applications", "bureau", "servicing", "holdout"])

### Issue log
Every confirmed problem is recorded with its file, column, number of affected rows,
the action taken (fix / flag / drop) and the reason. At the end of the audit this
becomes the issue table in `findings.md`.

In [19]:
ISSUES = {}
ISSUES_PATH = Path("../reports/audit_issues.csv")

def log_issue(file, column, issue, rows, action, reason):
    """Record one data problem. Re-running replaces the entry instead of duplicating it."""
    ISSUES[(file, column, issue)] = {"file": file, "column": column, "issue": issue,
                                     "rows": int(rows), "action": action, "reason": reason}

def save_issues():
    """Write the issue log to reports/ so it survives restarts and goes into findings.md."""
    table = pd.DataFrame(ISSUES.values())
    table.to_csv(ISSUES_PATH, index=False)
    return table

### Duplicates in each file
Two kinds of duplicates are counted:
- **Exact duplicates:** a whole row repeated.
- **Same id, different values:** the same loan recorded more than once with
  conflicting details, which is harder to resolve.

Duplicates would make a loan count twice in training and in the default rate.

In [3]:
def duplicate_report(df):
    """Two kinds of duplicates: identical copies, and same id with different values."""
    exact = df.duplicated().sum()                    # whole row repeated
    conflicting = df.drop_duplicates()["id"].duplicated().sum()  # same id, values differ
    return {"rows": len(df), "unique_ids": df["id"].nunique(),
            "exact_duplicates": int(exact), "same_id_different_values": int(conflicting)}

for name, df in data.items():
    print(name, duplicate_report(df))

applications {'rows': 30600, 'unique_ids': 30000, 'exact_duplicates': 0, 'same_id_different_values': 600}
bureau {'rows': 30000, 'unique_ids': 30000, 'exact_duplicates': 0, 'same_id_different_values': 0}
servicing {'rows': 30000, 'unique_ids': 30000, 'exact_duplicates': 0, 'same_id_different_values': 0}
holdout {'rows': 5000, 'unique_ids': 5000, 'exact_duplicates': 0, 'same_id_different_values': 0}


### What differs between conflicting copies?
For a few repeated ids, only the columns where the copies disagree are shown.
Knowing *what* differs tells us whether the copies are the same loan with
formatting differences, or genuinely different records.

In [4]:
def show_conflicts(df, n=3):
    """For a few ids that repeat with different values, show only the columns that differ."""
    deduped = df.drop_duplicates()
    ids = deduped.loc[deduped["id"].duplicated(), "id"].unique()[:n]
    for loan_id in ids:
        rows = deduped[deduped["id"] == loan_id]
        diff_cols = rows.columns[rows.nunique(dropna=False) > 1].tolist()
        print(f"\nid {loan_id} differs in: {diff_cols}")
        display(rows[["id"] + diff_cols])

show_conflicts(apps)


id 8acbab3c06 differs in: ['emp_title', 'dti']


,id,emp_title,dti
108,8acbab3c06,Hillstone,15.53
1199,8acbab3c06,HILLSTONE,15.54



id ddefd45498 differs in: ['emp_title', 'dti']


,id,emp_title,dti
429,ddefd45498,Safety Manager,15.36
1214,ddefd45498,SAFETY MANAGER,15.37



id 01edeb5c98 differs in: ['emp_title', 'dti']


,id,emp_title,dti
1083,01edeb5c98,Legal Negotiator,14.49
1506,01edeb5c98,LEGAL NEGOTIATOR,14.50


### Which columns differ, across all repeated ids?
Before deciding, we check that the pattern seen in the examples holds for every
repeated id. If an important field such as `defaulted` or `loan_amnt` ever
differed, the copies could not simply be merged.

In [7]:
def conflict_summary(df):
    """For every id that repeats with different values, count which columns differ."""
    deduped = df.drop_duplicates()
    dup = deduped[deduped["id"].duplicated(keep=False)]   # all copies of repeated ids
    counts = {}
    for loan_id, rows in dup.groupby("id"):
        for col in rows.columns[rows.nunique(dropna=False) > 1]:
            counts[col] = counts.get(col, 0) + 1
    print("Repeated ids:", dup["id"].nunique())
    return pd.Series(counts, dtype=int).sort_values(ascending=False)

conflict_summary(apps)

Repeated ids: 600


dti          600
emp_title    528
dtype: int64

In [8]:
dup = apps.drop_duplicates()
dup = dup[dup["id"].duplicated(keep=False)]
g = dup.groupby("id")

case_only = g["emp_title"].agg(lambda s: s.str.lower().nunique(dropna=False) == 1)
print("emp_title differs only by case:", case_only.all())
print("Largest dti difference:", (g["dti"].max() - g["dti"].min()).max())

emp_title differs only by case: True
Largest dti difference: 0.010000000000005116


### Decision: drop duplicate ids (keep first)
600 ids appear twice in `applications.csv`. The copies differ only in `emp_title`
capitalization (528 pairs) and `dti` by at most 0.01 (600 pairs); the label and
amounts never differ. They are the same loan entered twice, so the second copy is
dropped. Keeping both would double-count these loans.
This leaves 30,000 applications, matching the bureau and servicing files.

In [20]:
before = len(apps)
apps = apps.drop_duplicates(subset="id", keep="first").reset_index(drop=True)
print(f"applications: {before} → {len(apps)} rows")

log_issue(
    file="applications", column="id",
    issue="600 ids appear twice; copies differ only in emp_title capitalization (528) and dti by ≤0.01 (600)",
    rows=600,
    action="drop duplicate (keep first)",
    reason="Same loan entered twice; keeping both would double-count it. "
           "Differences are formatting/rounding, and the label and amounts never differ",
)

applications: 30000 → 30000 rows


### Do the three files cover the same loans?
Compares the sets of ids between applications, bureau and servicing. A loan
missing from one file would lose its bureau data or its servicing check when joined.

In [5]:
def coverage(df_a, df_b, name_a, name_b):
    """Ids present in one file but not the other."""
    a, b = set(df_a["id"]), set(df_b["id"])
    print(f"{name_a} ids missing from {name_b}: {len(a - b)}")
    print(f"{name_b} ids missing from {name_a}: {len(b - a)}")

coverage(apps, bureau, "applications", "bureau")
coverage(apps, serv, "applications", "servicing")
coverage(bureau, serv, "bureau", "servicing")

applications ids missing from bureau: 0
bureau ids missing from applications: 0
applications ids missing from servicing: 0
servicing ids missing from applications: 0
bureau ids missing from servicing: 0
servicing ids missing from bureau: 0


### Does the holdout overlap with training?
Holdout loans are from 2016 and should not appear in the training files.
Any overlap would put the same loan in both training and final scoring.

In [6]:
overlap = set(hold["id"]) & set(apps["id"])
print("Holdout ids also in applications:", len(overlap))

Holdout ids also in applications: 0


## A1.2 Formats

### Date formats
Each date value is reduced to its "shape" (letters → `A`, digits → `9`), so
`Dec-2013` becomes `AAA-9999` and `2013-12-01` becomes `9999-99-99`. More than one
shape in a column means mixed date formats, which can be parsed incorrectly or
silently, for example by swapping day and month.

In [10]:
def pattern_counts(s):
    """Reduce each value to its 'shape': letters become A, digits become 9."""
    shapes = (s.dropna().astype(str)
              .str.replace(r"[A-Za-z]", "A", regex=True)
              .str.replace(r"\d", "9", regex=True))
    return shapes.value_counts()

DATE_COLS = {
    "applications": ["issue_d"],
    "bureau": ["earliest_cr_line"],
    "servicing": ["last_pymnt_d", "last_credit_pull_d"],
    "holdout": ["issue_d", "earliest_cr_line"],
}

for name, cols in DATE_COLS.items():
    for col in cols:
        print(f"\n--- {name}.{col} ---")
        print(pattern_counts(data[name][col] if name != "applications" else apps[col]))


--- applications.issue_d ---
issue_d
AAA-9999      28800
9999-99-99      615
99/9999         585
Name: count, dtype: int64

--- bureau.earliest_cr_line ---
earliest_cr_line
AAA-9999      28800
9999-99-99      607
99/9999         593
Name: count, dtype: int64

--- servicing.last_pymnt_d ---
last_pymnt_d
AAA-9999    29979
Name: count, dtype: int64

--- servicing.last_credit_pull_d ---
last_credit_pull_d
AAA-9999    29999
Name: count, dtype: int64

--- holdout.issue_d ---
issue_d
AAA-9999    5000
Name: count, dtype: int64

--- holdout.earliest_cr_line ---
earliest_cr_line
AAA-9999    5000
Name: count, dtype: int64


### Case and spacing variants in categories
Counts distinct values before and after removing surrounding spaces and lowercasing.
If normalizing reduces the count, the same category is written in more than one way
(e.g. `RENT` vs `rent `), which would split one group into several.
The holdout is checked too, because it may contain variants that training does not.

In [11]:
TEXT_COLS = ["term", "grade", "sub_grade", "purpose", "application_type",
             "emp_length", "home_ownership", "verification_status", "addr_state"]

def variant_check(df, cols):
    """Distinct values before vs after normalizing case and spaces."""
    rows = []
    for c in cols:
        raw = df[c].dropna().astype(str)
        rows.append({"column": c, "distinct_raw": raw.nunique(),
                     "distinct_normalized": raw.str.strip().str.lower().nunique()})
    out = pd.DataFrame(rows)
    out["has_variants"] = out["distinct_raw"] > out["distinct_normalized"]
    return out

print("applications"); display(variant_check(apps, TEXT_COLS))
print("holdout"); display(variant_check(hold, TEXT_COLS))

applications


,column,distinct_raw,distinct_normalized,has_variants
0,term,2,2,False
1,grade,7,7,False
2,sub_grade,35,35,False
3,purpose,13,13,False
4,application_type,2,2,False
5,emp_length,11,11,False
6,home_ownership,3,3,False
7,verification_status,3,3,False
8,addr_state,154,61,True


holdout


,column,distinct_raw,distinct_normalized,has_variants
0,term,2,2,False
1,grade,7,7,False
2,sub_grade,35,35,False
3,purpose,12,12,False
4,application_type,2,2,False
5,emp_length,11,11,False
6,home_ownership,3,3,False
7,verification_status,3,3,False
8,addr_state,50,50,False


### Inspect the variants
For each column flagged above, lists every raw spelling grouped under its
normalized value, with counts, to decide how to standardize them.

In [12]:
def show_variants(df, col):
    """Group raw spellings by their normalized form; show groups with more than one spelling."""
    s = df[col].dropna().astype(str)
    groups = s.groupby(s.str.strip().str.lower()).agg(lambda x: x.value_counts().to_dict())
    return groups[groups.apply(len) > 1]

# example: show_variants(hold, "home_ownership")

In [15]:
norm = apps["addr_state"].dropna().astype(str).str.strip().str.lower()
print(norm[~norm.str.fullmatch(r"[a-z]{2}")].value_counts())

addr_state
california        148
texas              93
new york           77
florida            56
georgia            44
pennsylvania       43
new jersey         40
ohio               36
illinois           35
north carolina     32
michigan           24
virginia           24
Name: count, dtype: int64


### Mixed date formats
`issue_d` and `earliest_cr_line` each contain three formats: `Dec-2013` (28,800 rows),
`2013-12-01` (~610) and `12/2013` (~590). Servicing and holdout dates use only the
first. Each format is parsed with its own explicit rule (not pandas' automatic
guessing, which can silently swap day and month), the results are checked for
failures and implausible values, and all dates are stored at month precision.

In [13]:
DATE_FORMATS = {
    r"[A-Za-z]{3}-\d{4}": "%b-%Y",     # Dec-2013
    r"\d{4}-\d{2}-\d{2}": "%Y-%m-%d",  # 2013-12-01
    r"\d{2}/\d{4}": "%m/%Y",           # 12/2013
}

def parse_mixed_dates(s):
    """Parse each known date shape with its own format; anything else becomes NaT."""
    s = s.astype(str).str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")
    for pattern, fmt in DATE_FORMATS.items():
        mask = s.str.fullmatch(pattern)
        out[mask] = pd.to_datetime(s[mask], format=fmt, errors="coerce")
    return out.dt.to_period("M").dt.to_timestamp()  # month precision

In [14]:
def check_dates(raw, parsed, name):
    """Report unmatched shapes, failed parses, and the parsed range."""
    raw = raw.astype(str).str.strip()
    known = pd.Series(False, index=raw.index)
    for pattern in DATE_FORMATS:
        known |= raw.str.fullmatch(pattern)
    print(f"\n{name}")
    print("  values matching no known format:", int((~known & raw.ne("nan")).sum()))
    print("  matched a format but failed to parse:", int((known & parsed.isna()).sum()))
    print("  range:", parsed.min().date(), "to", parsed.max().date())

    iso = raw[raw.str.fullmatch(r"\d{4}-\d{2}-\d{2}")]
    print("  ISO dates, day part values:", iso.str[-2:].value_counts().to_dict())
    print("  ISO dates, middle part > 12:", int((iso.str[5:7].astype(int) > 12).sum()))

apps["issue_dt"] = parse_mixed_dates(apps["issue_d"])
bureau["earliest_cr_dt"] = parse_mixed_dates(bureau["earliest_cr_line"])
check_dates(apps["issue_d"], apps["issue_dt"], "applications.issue_d")
check_dates(bureau["earliest_cr_line"], bureau["earliest_cr_dt"], "bureau.earliest_cr_line")


applications.issue_d
  values matching no known format: 0
  matched a format but failed to parse: 0
  range: 2013-01-01 to 2015-12-01
  ISO dates, day part values: {'01': 615}
  ISO dates, middle part > 12: 0

bureau.earliest_cr_line
  values matching no known format: 0
  matched a format but failed to parse: 0
  range: 1951-01-01 to 2017-11-01
  ISO dates, day part values: {'01': 607}
  ISO dates, middle part > 12: 0


### Date results and address state
Both date columns parse fully with month precision (no unmatched values, no failed
parses, ISO dates are all day 01, no day/month swaps). However, some `earliest_cr_line`
values fall after the loan was issued, which is impossible and is counted below.
`addr_state` mixes two-letter codes, codes in different cases and full state names;
all are standardized to upper-case two-letter codes.

In [16]:
timeline = apps[["id", "issue_dt"]].merge(bureau[["id", "earliest_cr_dt"]], on="id")
after_issue = timeline["earliest_cr_dt"] > timeline["issue_dt"]
print("earliest_cr_line after issue_d:", int(after_issue.sum()))
display(timeline[after_issue].head())

earliest_cr_line after issue_d: 300


,id,issue_dt,earliest_cr_dt
27,7d1f6edf3c,2013-02-01,2013-11-01
51,13d470c317,2015-05-01,2016-12-01
78,2ba0ab5410,2015-02-01,2017-01-01
86,68732d8749,2015-06-01,2016-09-01
108,8acbab3c06,2013-01-01,2013-04-01


In [17]:
STATE_NAMES = {
    "alabama": "AL", "alaska": "AK", "arizona": "AZ", "arkansas": "AR",
    "california": "CA", "colorado": "CO", "connecticut": "CT", "delaware": "DE",
    "district of columbia": "DC", "florida": "FL", "georgia": "GA", "hawaii": "HI",
    "idaho": "ID", "illinois": "IL", "indiana": "IN", "iowa": "IA", "kansas": "KS",
    "kentucky": "KY", "louisiana": "LA", "maine": "ME", "maryland": "MD",
    "massachusetts": "MA", "michigan": "MI", "minnesota": "MN", "mississippi": "MS",
    "missouri": "MO", "montana": "MT", "nebraska": "NE", "nevada": "NV",
    "new hampshire": "NH", "new jersey": "NJ", "new mexico": "NM", "new york": "NY",
    "north carolina": "NC", "north dakota": "ND", "ohio": "OH", "oklahoma": "OK",
    "oregon": "OR", "pennsylvania": "PA", "rhode island": "RI", "south carolina": "SC",
    "south dakota": "SD", "tennessee": "TN", "texas": "TX", "utah": "UT",
    "vermont": "VT", "virginia": "VA", "washington": "WA", "west virginia": "WV",
    "wisconsin": "WI", "wyoming": "WY",
}
VALID_CODES = set(STATE_NAMES.values())

def clean_state(s):
    """Map full names to codes and upper-case codes; anything unrecognized becomes NaN."""
    norm = s.astype(str).str.strip().str.lower()
    mapped = norm.map(STATE_NAMES).fillna(norm.str.upper())
    return mapped.where(mapped.isin(VALID_CODES))

raw_state = apps["addr_state"].astype(str).str.strip()
apps["addr_state_clean"] = clean_state(apps["addr_state"])

print("written as full name:", int(raw_state.str.lower().isin(STATE_NAMES).sum()))
print("code not in upper case:", int((raw_state.str.fullmatch(r"[A-Za-z]{2}")
                                      & (raw_state != raw_state.str.upper())).sum()))
print("unrecognized after cleaning:", int(apps["addr_state_clean"].isna().sum()))
print("distinct states after cleaning:", apps["addr_state_clean"].nunique())

written as full name: 652
code not in upper case: 1311
unrecognized after cleaning: 0
distinct states after cleaning: 49


In [21]:
log_issue("applications", "issue_d",
          "Mixed date formats: Dec-2013 (28,800), 2013-12-01 (615), 12/2013 (585)",
          rows=1200, action="fix",
          reason="Parsed each format explicitly to month precision; all parse, no day/month swaps")

log_issue("bureau", "earliest_cr_line",
          "Mixed date formats: Dec-2013 (28,800), 2013-12-01 (607), 12/2013 (593)",
          rows=1200, action="fix",
          reason="Same rule as issue_d; all values parse")

log_issue("applications", "addr_state",
          "State written as full name (652) or code not in upper case (1,311)",
          rows=652 + 1311, action="fix",
          reason="Mapped names to two-letter codes and upper-cased; 0 unrecognized. "
                 "Kept for fairness analysis; likely excluded from the model as a location proxy")

In [22]:
save_issues()

,file,column,issue,rows,action,reason
0,applications,id,600 ids appear twice; copies differ only in em...,600,drop duplicate (keep first),Same loan entered twice; keeping both would do...
1,applications,issue_d,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Parsed each format explicitly to month precisi...
2,bureau,earliest_cr_line,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Same rule as issue_d; all values parse
3,applications,addr_state,State written as full name (652) or code not i...,1963,fix,Mapped names to two-letter codes and upper-cas...


### A1.2 summary
- `issue_d` and `earliest_cr_line`: 1,200 rows each in non-standard formats → parsed
  explicitly to month precision, all valid.
- `addr_state`: 1,963 rows in non-standard form (full names or lower case) →
  standardized to two-letter codes.
- Other categories (term, grade, purpose, emp_length, home_ownership,
  verification_status, application_type) have no case or spacing variants.
- Servicing and holdout dates use a single format.
- Noted for A1.4: 300 loans have `earliest_cr_line` after `issue_d` (impossible).

## A1.3 Impossible values and unit errors
Each numeric field is checked against the range of values that is physically or
contractually possible (e.g. loans of 1,000–40,000 USD, FICO 300–850, no negative
balances or utilization). Values outside these ranges are counted, then investigated
to separate unit errors (e.g. income entered in thousands) from data-entry errors.
Genuine but extreme values are left for A3 (outlier treatment).

In [23]:
train = apps.merge(bureau, on="id", how="inner")

RULES = {   # column: (lowest possible, highest possible); None = no limit
    "loan_amnt": (1000, 40000), "int_rate": (0, 40), "installment": (0.01, None),
    "annual_inc": (1000, None), "dti": (0, 100),
    "fico_range_low": (300, 850), "fico_range_high": (300, 850),
    "revol_util": (0, None), "bc_util": (0, None),
    "revol_bal": (0, None), "tot_cur_bal": (0, None), "avg_cur_bal": (0, None),
    "total_rev_hi_lim": (0, None), "tot_hi_cred_lim": (0, None),
    "inq_last_6mths": (0, None), "delinq_2yrs": (0, None), "pub_rec": (0, None),
    "pub_rec_bankruptcies": (0, None), "open_acc": (0, None), "total_acc": (0, None),
    "mort_acc": (0, None), "acc_open_past_24mths": (0, None),
    "num_tl_op_past_12m": (0, None), "mths_since_last_delinq": (0, None),
}

def range_violations(df, rules):
    """Count values below the lowest or above the highest possible value."""
    rows = []
    for col, (lo, hi) in rules.items():
        if col not in df:
            continue
        s = df[col]
        below = int((s < lo).sum()) if lo is not None else 0
        above = int((s > hi).sum()) if hi is not None else 0
        if below or above:
            rows.append({"column": col, "below_min": below, "above_max": above,
                         "min": s.min(), "max": s.max()})
    return pd.DataFrame(rows)

print("training (applications + bureau)"); display(range_violations(train, RULES))
print("holdout"); display(range_violations(hold, RULES))

training (applications + bureau)


,column,below_min,above_max,min,max
0,loan_amnt,0,280,1000.0,350000.00
1,annual_inc,899,0,11.5,7500000.00
2,dti,0,1,0.0,100.09
3,revol_util,295,0,-101.3,366.60
4,bc_util,290,0,-102.4,156.20


holdout


,column,below_min,above_max,min,max
0,dti,0,2,0.0,126.8


### Investigating out-of-range values
Each violation is tested against independent evidence before deciding a fix:
- `loan_amnt` above 40,000: is the monthly installment consistent with an amount
  ten times smaller?
- `annual_inc` below 1,000: would yearly loan payments exceed yearly income, and does
  ×1,000 bring these incomes in line with the rest?
- `dti` above 100: inspected directly.
- Negative `revol_util` / `bc_util`: does the absolute value match utilization
  recalculated from balance ÷ limit?

In [24]:
train["term_months"] = train["term"].str.extract(r"(\d+)", expand=False).astype(int)

def expected_installment(amount, rate_pct, months):
    """Standard loan payment formula: amount × r / (1 − (1 + r)^−n), r = monthly rate."""
    r = rate_pct / 100 / 12
    return amount * r / (1 - (1 + r) ** (-months))

train["inst_ratio"] = train["installment"] / expected_installment(
    train["loan_amnt"], train["int_rate"], train["term_months"])

big = train["loan_amnt"] > 40000
print("normal rows, installment ÷ expected:"); print(train.loc[~big, "inst_ratio"].describe())
print("\nrows above 40,000, installment ÷ expected:"); print(train.loc[big, "inst_ratio"].describe())
print("\nmost common large amounts:"); print(train.loc[big, "loan_amnt"].value_counts().head(8))

normal rows, installment ÷ expected:
count    29720.000000
mean         0.999383
std          0.023598
min          0.100002
25%          1.000006
50%          1.000012
75%          1.000020
max          1.073606
Name: inst_ratio, dtype: float64

rows above 40,000, installment ÷ expected:
count    2.800000e+02
mean     1.000013e-01
std      9.995546e-07
min      1.000000e-01
25%      1.000006e-01
50%      1.000011e-01
75%      1.000017e-01
max      1.000054e-01
Name: inst_ratio, dtype: float64

most common large amounts:
loan_amnt
100000.0    24
120000.0    18
150000.0    17
200000.0    13
350000.0    11
240000.0    10
50000.0     10
180000.0     8
Name: count, dtype: int64


In [28]:
x10 = train["inst_ratio"].between(0.099, 0.101)
amount_if_fixed = train["loan_amnt"] / 10

print("×10 rows:", int(x10.sum()))
print("  amount > 40,000 (amount clearly wrong):", int((x10 & (train["loan_amnt"] > 40000)).sum()))
print("  amount/10 < 1,000 (installment must be wrong instead):",
      int((x10 & (amount_if_fixed < 1000)).sum()))
print("  undetermined (amount ≤ 40,000 and amount/10 ≥ 1,000):",
      int((x10 & (train["loan_amnt"] <= 40000) & (amount_if_fixed >= 1000)).sum()))
print("\ninstallment in ×10 rows vs others (median):",
      train.loc[x10, "installment"].median(), "vs", train.loc[~x10, "installment"].median())

×10 rows: 300
  amount > 40,000 (amount clearly wrong): 280
  amount/10 < 1,000 (installment must be wrong instead): 0
  undetermined (amount ≤ 40,000 and amount/10 ≥ 1,000): 20

installment in ×10 rows vs others (median): 399.46500000000003 vs 382.545


### Decision: loan amounts recorded ten times too large
300 loans have an installment exactly one-tenth of what their stated amount requires
(installment ÷ expected = 0.100). For 280 the amount exceeds the 40,000 limit; 20 are
within the limit and would be missed by a range check. Their installments are typical
and no loan shows evidence of a wrong installment, so the amount is the corrupted field.
All 300 are divided by 10 and flagged in `loan_amnt_corrected`.

In [29]:
x10 = train["inst_ratio"].between(0.099, 0.101)

train["loan_amnt_corrected"] = x10.astype(int)          # 1 = amount was fixed
train.loc[x10, "loan_amnt"] = train.loc[x10, "loan_amnt"] / 10

print("rows fixed:", int(x10.sum()))
print("loan_amnt range after fix:", train["loan_amnt"].min(), "to", train["loan_amnt"].max())

log_issue("applications", "loan_amnt",
          "Amount recorded 10× too large: 280 above the 40,000 limit + 20 within it",
          rows=int(x10.sum()), action="fix (÷10) + flag",
          reason="Installment is exactly 1/10 of the payment the stated amount requires "
                 "(ratio 0.100); installments are typical and no loan shows a wrong installment, "
                 "so the amount is the corrupted field. 20 errors were within the valid range "
                 "and found only by the installment cross-check")
save_issues()

rows fixed: 300
loan_amnt range after fix: 1000.0 to 35000.0


,file,column,issue,rows,action,reason
0,applications,id,600 ids appear twice; copies differ only in em...,600,drop duplicate (keep first),Same loan entered twice; keeping both would do...
1,applications,issue_d,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Parsed each format explicitly to month precisi...
2,bureau,earliest_cr_line,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Same rule as issue_d; all values parse
3,applications,addr_state,State written as full name (652) or code not i...,1963,fix,Mapped names to two-letter codes and upper-cas...
4,applications,loan_amnt,Amount recorded 10× too large: 280 above the 4...,300,fix (÷10) + flag,Installment is exactly 1/10 of the payment the...


In [25]:
low = train["annual_inc"] < 1000
yearly_payments = train["installment"] * 12

print("low incomes:"); print(train.loc[low, "annual_inc"].describe())
print("\nshare of low rows where yearly payments exceed income (raw):",
      round((yearly_payments[low] > train.loc[low, "annual_inc"]).mean(), 3))
print("median income, normal rows:", train.loc[~low, "annual_inc"].median())
print("median income, low rows × 1,000:", (train.loc[low, "annual_inc"] * 1000).median())
print("median payment share of income, normal rows:",
      round((yearly_payments[~low] / train.loc[~low, "annual_inc"]).median(), 3))
print("median payment share, low rows × 1,000:",
      round((yearly_payments[low] / (train.loc[low, "annual_inc"] * 1000)).median(), 3))

low incomes:
count    899.000000
mean      73.695217
std       43.331007
min       11.500000
25%       45.000000
50%       63.500000
75%       91.500000
max      470.000000
Name: annual_inc, dtype: float64

share of low rows where yearly payments exceed income (raw): 1.0
median income, normal rows: 65000.0
median income, low rows × 1,000: 63500.0
median payment share of income, normal rows: 0.074
median payment share, low rows × 1,000: 0.077


### Income entered in thousands
899 incomes are below 1,000 (median 63.5). For all of them, yearly loan payments would
exceed the stated income, which is impossible. Multiplied by 1,000, their median income
(63,500) and payment share (7.7%) match the other loans (65,000 and 7.4%).
Before fixing, we confirm a clean gap between these values and real incomes, so the
1,000 cut-off cannot catch a genuine income. Values below 1,000 are then ×1,000,
the same rule given in the brief, and applied at scoring time too.

In [30]:
inc = train["annual_inc"]
print("largest income below 1,000:", inc[inc < 1000].max())
print("incomes between 470 and 1,000:", int(inc.between(470, 1000, inclusive="neither").sum()))
print("incomes between 1,000 and 10,000:", int(inc.between(1000, 10000).sum()))
print("smallest incomes of 1,000 or more:", sorted(inc[inc >= 1000].unique())[:5])

largest income below 1,000: 470.0
incomes between 470 and 1,000: 0
incomes between 1,000 and 10,000: 19
smallest incomes of 1,000 or more: [np.float64(7500.0), np.float64(7900.0), np.float64(8000.0), np.float64(8616.0), np.float64(8800.0)]


In [31]:
low = train["annual_inc"] < 1000

train["annual_inc_corrected"] = low.astype(int)          # 1 = income was fixed
train.loc[low, "annual_inc"] = train.loc[low, "annual_inc"] * 1000

print("rows fixed:", int(low.sum()))
print("annual_inc range after fix:", train["annual_inc"].min(), "to", train["annual_inc"].max())

log_issue("applications", "annual_inc",
          "Income entered in thousands (values below 1,000, median 63.5)",
          rows=int(low.sum()), action="fix (×1,000) + flag",
          reason="Raw values make yearly payments exceed income in 100% of these rows; ×1,000 gives "
                 "median 63,500 and payment share 7.7%, matching other loans (65,000, 7.4%). "
                 "Clean gap between these values and real incomes")
save_issues()

rows fixed: 899
annual_inc range after fix: 7500.0 to 7500000.0


,file,column,issue,rows,action,reason
0,applications,id,600 ids appear twice; copies differ only in em...,600,drop duplicate (keep first),Same loan entered twice; keeping both would do...
1,applications,issue_d,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Parsed each format explicitly to month precisi...
2,bureau,earliest_cr_line,"Mixed date formats: Dec-2013 (28,800), 2013-12...",1200,fix,Same rule as issue_d; all values parse
3,applications,addr_state,State written as full name (652) or code not i...,1963,fix,Mapped names to two-letter codes and upper-cas...
4,applications,loan_amnt,Amount recorded 10× too large: 280 above the 4...,300,fix (÷10) + flag,Installment is exactly 1/10 of the payment the...
5,applications,annual_inc,"Income entered in thousands (values below 1,00...",899,"fix (×1,000) + flag",Raw values make yearly payments exceed income ...


In [26]:
cols = ["dti", "annual_inc", "installment", "application_type", "verification_status"]
print("dti > 40 — training:", int((train["dti"] > 40).sum()), "| holdout:", int((hold["dti"] > 40).sum()))
display(train.loc[train["dti"] > 40, cols].sort_values("dti", ascending=False).head(10))
display(hold.loc[hold["dti"] > 40, cols].sort_values("dti", ascending=False))

dti > 40 — training: 4 | holdout: 41


,dti,annual_inc,installment,application_type,verification_status
14637,100.09,27000.0,510.05,Joint App,Source Verified
4091,48.91,38500.0,778.06,Joint App,Source Verified
1603,47.31,40000.0,768.19,Joint App,Verified
24422,44.40,30000.0,461.90,Joint App,Verified


,dti,annual_inc,installment,application_type,verification_status
3056,126.80,12000.0,1174.91,Joint App,Not Verified
3402,124.13,12000.0,545.68,Joint App,Source Verified
4524,58.94,20000.0,587.70,Joint App,Source Verified
3762,52.42,25000.0,167.85,Joint App,Verified
1306,51.57,42000.0,178.24,Joint App,Verified
3669,48.60,51000.0,505.34,Joint App,Verified
2286,48.42,38000.0,301.15,Joint App,Verified
1694,48.20,42000.0,596.44,Joint App,Verified
1269,47.43,110000.0,584.94,Individual,Not Verified
3129,47.26,30272.0,186.48,Individual,Source Verified


In [27]:
recomputed = train["revol_bal"] / train["total_rev_hi_lim"] * 100   # inf when limit is 0
neg_r, neg_b = train["revol_util"] < 0, train["bc_util"] < 0

def close(a, b, tol=5):
    """Share of rows where a and b are within tol percentage points."""
    return round(((a - b).abs() <= tol).mean(), 3)

print("normal revol_util close to recomputed (baseline):", close(train.loc[~neg_r, "revol_util"], recomputed[~neg_r]))
print("negative revol_util: |value| close to recomputed:", close(train.loc[neg_r, "revol_util"].abs(), recomputed[neg_r]))
print("rows negative in both revol_util and bc_util:", int((neg_r & neg_b).sum()))

high = train["revol_util"] > 150
display(train.loc[high, ["revol_util", "revol_bal", "total_rev_hi_lim"]].assign(recomputed=recomputed[high]))

normal revol_util close to recomputed (baseline): 0.961
negative revol_util: |value| close to recomputed: 0.973
rows negative in both revol_util and bc_util: 1


,revol_util,revol_bal,total_rev_hi_lim,recomputed
28306,366.6,2566.0,700.0,366.571429
